# NICU mortality: 24-hour LSTM autoencoder

Research prototype only; not a clinical risk calculator. The score is a reconstruction error, not a probability of death. This is a documented approximation, not an exact reproduction of the article. The cohort can include patients older than 28 days.

Colab does **not** connect to PostgreSQL. Prepare the single minimized NPZ locally using `python -m src.mortality_exchange`. Do not upload the original dataset or the local audit NPZ. Confirm dataset-license and institutional approval before using Drive or Colab.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/jcatumba/neonatal-sepsis-lstm.git'
REPO_REF = 'main'  # For reproducibility, replace with a published commit SHA.
REPO_DIR = Path('/content/neonatal-sepsis-lstm')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_REF], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPO_URL:
        raise RuntimeError('Existing clone has a different origin; use a fresh runtime.')
    head = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
    expected = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', f'{REPO_REF}^{{commit}}'], text=True).strip()
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True)
    if head != expected or dirty:
        raise RuntimeError('Existing clone is dirty or at another revision; use a fresh runtime.')
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'requirements-colab.txt'], check=True)
from importlib.metadata import PackageNotFoundError, version
print('Python:', sys.version.split()[0], 'PyTorch:', version('torch'))
try:
    print('TensorFlow:', version('tensorflow'))
except PackageNotFoundError:
    print('TensorFlow: not installed or required')
GIT_REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Git revision:', GIT_REVISION)

Python: 3.13.18 PyTorch: 2.8.0
TensorFlow: 2.21.0
Git revision: cd825da8df5c2e997ff736cf343ac0f0e1d1fec1


## Configuration and data access
The default requests the NPZ path after mounting Drive. Use `upload` for a small local NPZ or `demo` for synthetic data. Set `FAST_DEV_RUN=True` for a two-epoch smoke test; do not interpret smoke or synthetic results as study findings.

In [2]:
import hashlib
import io
import json
import shutil
from datetime import datetime, timezone
import numpy as np
import torch
from src.mortality_exchange import demo_exchange, transform_new_sequence, validate_exchange
from src.mortality_model import MortalityAutoencoder, MortalityConfig
from src.mortality_eval import reconstruction_scores
from src.mortality_pipeline import run_mortality_experiment

DATA_SOURCE = 'upload'  # 'drive', 'upload', or 'demo'
DRIVE_DATASET_PATH = 'DeepLearning'  # Leave empty to enter the path when prompted.
FAST_DEV_RUN = False
COPY_TO_DRIVE = False
DRIVE_OUTPUT_DIR = ''  # If copying, enter a parent folder when prompted.
config = MortalityConfig(seed=42, batch_size=32, epochs=10, learning_rate=0.0001,
                         bootstrap_samples=1000, fast_dev_run=FAST_DEV_RUN,
                         device_preference='cuda')
print('Device:', config.device)

Device: cpu


## Load and validate the exchange file
The file contains only prepared training, validation and test arrays, labels needed for evaluation, a pseudonymous test-patient grouping, and training preprocessing parameters. It still contains sensitive derived clinical data.

In [3]:
if DATA_SOURCE == 'demo':
    arrays = demo_exchange(config.seed)
    source = 'synthetic'
    dataset_sha256 = None
elif DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    dataset_path = DRIVE_DATASET_PATH.strip() or input('Full path to the approved mortality_colab.npz in Drive: ').strip()
    if not dataset_path:
        raise ValueError('A Drive dataset path is required.')
    payload = Path(dataset_path).read_bytes()
    with np.load(io.BytesIO(payload), allow_pickle=False) as archive:
        arrays = validate_exchange({key: archive[key] for key in archive.files})
    dataset_sha256 = hashlib.sha256(payload).hexdigest()
    source = 'prepared'
elif DATA_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one prepared NPZ file.')
    name, payload = next(iter(uploaded.items()))
    if Path(name).suffix.lower() != '.npz':
        raise ValueError('The uploaded file must be an NPZ.')
    with np.load(io.BytesIO(payload), allow_pickle=False) as archive:
        arrays = validate_exchange({key: archive[key] for key in archive.files})
    dataset_sha256 = hashlib.sha256(payload).hexdigest()
    source = 'prepared'
else:
    raise ValueError('DATA_SOURCE must be drive, upload, or demo.')
for split in ('train', 'validation', 'test'):
    print(split, 'admissions:', len(arrays[f'{split}_x']))
print('Inputs: 10 bins, 3 biomarkers, 3 observation masks.')

Saving mortality_colab.npz to mortality_colab.npz
train admissions: 1499
validation admissions: 318
test admissions: 320
Inputs: 10 bins, 3 biomarkers, 3 observation masks.


## Train and evaluate
The 64/32 encoder and 32/64 decoder train on all training admissions with observed-only MSE. Observed-only MAE is scored per admission. Youden's threshold is chosen on validation labels; test metrics and 95% patient-grouped bootstrap intervals use the held-out test set. Intervals condition on the trained model and fixed threshold.

In [4]:
run_root = Path('/content/outputs/mortality')
run_root.mkdir(parents=True, exist_ok=True)
run_dir = run_root / datetime.now(timezone.utc).strftime('run-%Y%m%dT%H%M%S%fZ')
metrics = run_mortality_experiment(arrays, config, run_dir, source=source,
                                   provenance={'git_revision': GIT_REVISION, 'dataset_sha256': dataset_sha256})
print(json.dumps(metrics, indent=2))
print('Artifacts:', run_dir)
print('Scores are not calibrated death probabilities or clinical recommendations.')

CUDA unavailable; training on CPU.
Epoch 1: train=0.027985, validation=0.021044
Epoch 2: train=0.017570, validation=0.013630
Epoch 3: train=0.013321, validation=0.012085
Epoch 4: train=0.012193, validation=0.011114
Epoch 5: train=0.011189, validation=0.010100
Epoch 6: train=0.010056, validation=0.008807
Epoch 7: train=0.008751, validation=0.007717
Epoch 8: train=0.008005, validation=0.007163
Epoch 9: train=0.007359, validation=0.006627
Epoch 10: train=0.006825, validation=0.006312
{
  "threshold": 0.03902684897184372,
  "threshold_source": "validation_youden",
  "threshold_comparison": "score >= threshold",
  "score": "mean_absolute_error_over_observed_biomarkers",
  "point_metrics": {
    "auroc": 0.6212084148727985,
    "average_precision": 0.1775930819401818,
    "sensitivity": 0.8571428571428571,
    "specificity": 0.339041095890411,
    "precision": 0.11059907834101383,
    "accuracy": 0.384375,
    "confusion_matrix": [
      [
        99,
        193
      ],
      [
        4,


## Persist artifacts
By default outputs stay in the temporary Colab runtime; download them before disconnecting. Optional Drive copying requires separate authorization. Trained weights, aggregate metrics and preprocessing parameters can still disclose information about the training data; handle them as sensitive.

In [5]:
def save_to_drive_if_enabled(source_dir, destination_parent):
    if not COPY_TO_DRIVE:
        return None
    from google.colab import drive
    drive.mount('/content/drive')
    parent_text = destination_parent.strip() or input('Approved Drive output parent folder: ').strip()
    if not parent_text:
        raise ValueError('A Drive output parent folder is required.')
    parent = Path(parent_text).resolve()
    if not parent.is_relative_to(Path('/content/drive').resolve()) or not parent.is_dir():
        raise ValueError('Output parent must be an existing folder on the mounted Drive.')
    destination = parent / source_dir.name
    shutil.copytree(source_dir, destination)
    return destination

copied = save_to_drive_if_enabled(run_dir, DRIVE_OUTPUT_DIR)
print('Drive copy:', copied if copied else 'disabled')
print('Saved:', sorted(path.name for path in run_dir.iterdir()))

Drive copy: disabled
Saved: ['best_model.pt', 'config.json', 'history.json', 'metrics.json', 'model_spec.json', 'pr_curve.png', 'preprocessor.json', 'provenance.json', 'roc_curve.png', 'training_curve.png', 'versions.json']


## Inference example with synthetic input
New inputs must first be binned into ten 2.4-hour intervals, using the last valid value in each bin and the source units documented in the report. Missing measurements are `NaN`. This demonstration is synthetic and does not print scores for any dataset patient.

In [6]:
metadata = json.loads((run_dir / 'preprocessor.json').read_text())
raw_example = np.full((10, 3), np.nan)
raw_example[0] = metadata['training_observation_medians']
sequence = transform_new_sequence(raw_example, metadata)[None, :, :]
model = MortalityAutoencoder()
model.load_state_dict(torch.load(run_dir / 'best_model.pt', map_location='cpu', weights_only=True))
example_score = float(reconstruction_scores(model, sequence)[0])
print('Synthetic example anomaly score:', example_score)
print('Above validation threshold:', example_score >= metrics['threshold'])

Synthetic example anomaly score: 0.020621193572878838
Above validation threshold: False
